# MSD Task09 Spleen: Preprocessing & Exploratory Data Analysis

> **Educational prototype only. Not for clinical diagnostic use.**

This notebook handles the **preprocessing stage** of the spleen 3D CT segmentation pipeline:
1. Discovers and validates raw NIfTI CT scans and binary spleen segmentation masks.
2. Analyzes volume dimensions, voxel spacing, and spleen voxel distributions.
3. Configures and verifies the deterministic MONAI transform pipeline (Orientation, Spacing, Intensity Scaling, Foreground Cropping).
4. Visualizes multi-planar slices (Axial, Coronal, Sagittal) with ground-truth spleen contours.
5. Sets up reproducible training and held-out validation splits.

In [ ]:
import os
import sys
from pathlib import Path
# Locate project root and ensure it is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import torch
import nibabel as nib
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

import monai
from monai.data import CacheDataset, DataLoader
from ml.segmentation.preprocessing import (
    IMAGE_KEY,
    LABEL_KEY,
    build_preprocessing_transforms,
    list_volume_pairs,
    validate_sample,
)

print('Project root:', project_root)
print('MONAI version:', monai.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
config_path = project_root / 'ml' / 'segmentation' / 'config.yaml'
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))

images_dir = project_root / config['dataset']['images_dir']
labels_dir = project_root / config['dataset']['labels_dir']

volume_pairs = list_volume_pairs(images_dir, labels_dir)
print(f'Total aligned NIfTI volume pairs available: {len(volume_pairs)}')
for i, p in enumerate(volume_pairs[:3]):
    print(f'[{i+1}] Image: {Path(p[IMAGE_KEY]).name} | Mask: {Path(p[LABEL_KEY]).name}')

In [ ]:
# Inspect the raw volume characteristics
sample_img = nib.load(volume_pairs[0][IMAGE_KEY])
sample_mask = nib.load(volume_pairs[0][LABEL_KEY])

img_data = sample_img.get_fdata()
mask_data = sample_mask.get_fdata()

spleen_voxels = int((mask_data > 0).sum())
total_voxels = mask_data.size
spleen_pct = 100.0 * spleen_voxels / total_voxels

print(f'Raw Image Shape: {img_data.shape}')
print(f'Voxel Spacing (zooms): {sample_img.header.get_zooms()}')
print(f'Intensity Range (HU): [{img_data.min():.1f}, {img_data.max():.1f}]')
print(f'Total Spleen Voxels: {spleen_voxels:,} ({spleen_pct:.2f}% of raw volume)')

### Deterministic Preprocessing Pipeline

Standardizes the input across patients:
- **Orientation**: Reorients voxel axes to standard `RAS` coordinates.
- **Resampling**: Normalizes voxel spacing to `(1.5, 1.5, 2.0)` mm.
- **Windowing**: Restricts abdominal soft-tissue CT intensities to `[-175, 250]` HU and scales to `[0.0, 1.0]`.
- **Foreground Crop**: Crops out empty ambient background air around the patient's body.

In [ ]:
# Build deterministic transforms and create a test preprocessed cache
prep_transforms = build_preprocessing_transforms(config)
demo_dataset = CacheDataset(data=volume_pairs[:2], transform=prep_transforms, cache_rate=1.0, num_workers=0)

sample = demo_dataset[0]
validate_sample(sample)

image_t = sample[IMAGE_KEY]
label_t = sample[LABEL_KEY]

print('Preprocessed Image Tensor Shape:', tuple(image_t.shape))
print('Preprocessed Label Tensor Shape:', tuple(label_t.shape))
print('Preprocessed Label Voxels:', int(label_t.sum().item()))
print(f'Intensity min: {image_t.min():.3f}, max: {image_t.max():.3f}')

In [ ]:
# Visualize multi-planar cross-sections with spleen ground truth
img_np = image_t[0].cpu().numpy()
lbl_np = label_t[0].cpu().numpy()

# Locate spleen center of mass (X=Sagittal, Y=Coronal, Z=Axial)
spleen_coords = np.argwhere(lbl_np > 0)
if len(spleen_coords) > 0:
    cx, cy, cz = spleen_coords.mean(axis=0).astype(int)
else:
    cx, cy, cz = [s // 2 for s in img_np.shape]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Axial slice (along Z-axis)
axes[0].imshow(img_np[:, :, cz].T, cmap='gray', origin='lower')
axes[0].contour(lbl_np[:, :, cz].T, levels=[0.5], colors='red', linewidths=1.5)
axes[0].set_title(f'Axial (Slice {cz})')
axes[0].axis('off')

# Coronal slice (along Y-axis)
axes[1].imshow(img_np[:, cy, :].T, cmap='gray', origin='lower')
axes[1].contour(lbl_np[:, cy, :].T, levels=[0.5], colors='red', linewidths=1.5)
axes[1].set_title(f'Coronal (Slice {cy})')
axes[1].axis('off')

# Sagittal slice (along X-axis)
axes[2].imshow(img_np[cx, :, :].T, cmap='gray', origin='lower')
axes[2].contour(lbl_np[cx, :, :].T, levels=[0.5], colors='red', linewidths=1.5)
axes[2].set_title(f'Sagittal (Slice {cx})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Dataset Split Policy

We divide the 41 available volumes into disjoint splits:
- **Training set**: 16 volumes for robust prototype training.
- **Held-out validation set**: 4 strictly non-overlapping volumes.

In [ ]:
TRAIN_COUNT = 16
VAL_COUNT = 4

train_pairs = volume_pairs[:TRAIN_COUNT]
val_pairs = volume_pairs[TRAIN_COUNT : TRAIN_COUNT + VAL_COUNT]

train_names = {Path(p[IMAGE_KEY]).name for p in train_pairs}
val_names = {Path(p[IMAGE_KEY]).name for p in val_pairs}

assert train_names.isdisjoint(val_names), 'Overlap detected between splits!'
print(f'Training volumes ({len(train_pairs)}):', sorted(train_names))
print(f'Validation volumes ({len(val_pairs)}):', sorted(val_names))
print('Split verification: Disjoint split confirmed.')